# Harmony fine-tuning (Vision+Text) — `microsoft/Phi-4-reasoning-vision-15B`

This notebook fine-tunes **Phi-4-reasoning-vision-15B** for **image+text → text** using a Harmony-style JSONL where each example contains:

- `image_path`: local path to the image (e.g. chest X-ray)
- `messages`: chat turns (user/assistant)

We use **LoRA (PEFT)** on top of the base model.


In [1]:
# -----------------------------
# Hugging Face auth (required if model is gated)
# -----------------------------
from huggingface_hub import login

token_path = "/data/liangz2/openi/hf_token.txt"
with open(token_path, "r") as f:
    hf_token = f.readline().strip()

login(token=hf_token)
print("✅ Hugging Face login successful.")


✅ Hugging Face login successful.


In [2]:
# from huggingface_hub import hf_hub_download
# from pathlib import Path
# import re

# MODEL_ID = "microsoft/Phi-4-reasoning-vision-15B"

# src_path = hf_hub_download(
#     repo_id=MODEL_ID,
#     filename="modeling_phi4_visionr.py",
# )

# print("Downloaded source:", src_path)

# text = Path(src_path).read_text(encoding="utf-8")

# classes = re.findall(r"^class\s+(\w+)\((.*?)\):", text, flags=re.MULTILINE)
# print("\nClasses found:\n")
# for name, bases in classes:
#     print(f"{name}({bases})")

In [3]:
import os, json, csv
from dataclasses import dataclass
from typing import Any, Dict, List
import torch
from torch.utils.data import Dataset

from PIL import Image

from transformers import (
    AutoProcessor,
    AutoConfig,
    AutoModel,
    AutoModelForVision2Seq,
    TrainingArguments,
    Trainer,
)
from transformers.dynamic_module_utils import get_class_from_dynamic_module
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


In [4]:
# -----------------------------
# Config
# -----------------------------
MODEL_ID = "microsoft/Phi-4-reasoning-vision-15B"

JSONL_PATH = "/data/liangz2/openi/harmony_set/openi_cxr_harmony_rl.jsonl"  # update if needed
OUTPUT_DIR = "/data/liangz2/openi/Phi_4_reason"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LEN = 1536          # adjust for your GPU
SEED = 42
TRAIN_FRAC = 0.98

# LoRA
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# Generation (for quick sanity checks)
GEN_MAX_NEW_TOKENS = 128


In [5]:
# -----------------------------
# Load processor + config
# -----------------------------
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if getattr(processor, "tokenizer", None) is not None and processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

# -----------------------------
# Load actual model class
# -----------------------------
Phi4ForCausalLMV = get_class_from_dynamic_module(
    class_reference="modeling_phi4_visionr.Phi4ForCausalLMV",
    pretrained_model_name_or_path=MODEL_ID,
)

model = Phi4ForCausalLMV.from_pretrained(
    MODEL_ID,
    config=config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

print("✅ Phi-4-reasoning-vision-15B loaded successfully")
print("model type:", type(model))
print("has named_modules:", hasattr(model, "named_modules"))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Phi-4-reasoning-vision-15B loaded successfully
model type: <class 'transformers_modules.microsoft.Phi_hyphen_4_hyphen_reasoning_hyphen_vision_hyphen_15B.8e0aeaf8f5c28e5784c123e94c85fd1a919704f8.modeling_phi4_visionr.Phi4ForCausalLMV'>
has named_modules: True


In [6]:
# for name, module in model.named_modules():
#     if any(x in name for x in ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]):
#         print(name, type(module))

In [6]:
# -----------------------------
# Attach LoRA
# -----------------------------
target_modules = ["qkv_proj", "o_proj", "gate_up_proj", "down_proj"]

# uncomment to also adapt the vision tower 
# target_modules = [
#     "qkv_proj", "o_proj", "gate_up_proj", "down_proj",
#     "q_proj", "k_proj", "v_proj"
# ]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 111,411,200 || all params: 15,230,929,344 || trainable%: 0.7315


In [7]:
# -----------------------------
# Helpers: parse Harmony JSONL
# -----------------------------
def _extract_text_from_content(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        out = []
        for part in content:
            if isinstance(part, str):
                out.append(part)
            elif isinstance(part, dict):
                if "text" in part and isinstance(part["text"], str):
                    out.append(part["text"])
                elif "content" in part and isinstance(part["content"], str):
                    out.append(part["content"])
                else:
                    pass
        return "\n".join([x for x in out if x.strip()])
    if isinstance(content, dict):
        if "text" in content and isinstance(content["text"], str):
            return content["text"]
        if "content" in content and isinstance(content["content"], str):
            return content["content"]
        return json.dumps(content, ensure_ascii=False)
    return str(content)

def _normalize_messages(msgs):
    norm = []
    for m in msgs or []:
        if not isinstance(m, dict):
            continue
        role = (m.get("role") or "").strip().lower()
        if role not in {"system", "user", "assistant"}:
            role = "user"
        content = _extract_text_from_content(m.get("content"))
        norm.append({"role": role, "content": content})
    return norm


import json
from typing import Any, Dict, List

def _make_harder_assistant_target(ans: str) -> str:
    ans = str(ans).strip().upper()
    if ans == "TRUE":
        return "Verdict: TRUE\nReason: Supported by the evidence."
    elif ans == "FALSE":
        return "Verdict: FALSE\nReason: Contradicted by the evidence."
    else:
        raise ValueError(f"Unexpected answer label: {ans!r}")

def read_harmony_jsonl_with_images(jsonl_path: str) -> List[Dict[str, Any]]:
    """
    Reads Harmony JSONL with:
      - image_path
      - messages
      - ground_truth.answer ("TRUE"/"FALSE")

    If no assistant message exists, synthesize a harder assistant target:
      Verdict: TRUE/FALSE
      Reason: ...
    """
    rows: List[Dict[str, Any]] = []
    skipped_no_image = 0
    skipped_no_msgs = 0
    skipped_no_gt = 0

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            ex = json.loads(line)
            img_path = ex.get("image_path", None)
            msgs = ex.get("messages", None)

            if not img_path:
                skipped_no_image += 1
                continue
            if not isinstance(msgs, list) or len(msgs) == 0:
                skipped_no_msgs += 1
                continue

            msgs = _normalize_messages(msgs)

            if not any(m["role"] == "assistant" for m in msgs):
                gt = ex.get("ground_truth", {})
                ans = gt.get("answer") if isinstance(gt, dict) else None
                if not ans:
                    skipped_no_gt += 1
                    continue
                assistant_text = _make_harder_assistant_target(ans)
                msgs.append({"role": "assistant", "content": assistant_text})
            else:
                # If assistant already exists, optionally normalize TRUE/FALSE-only targets
                last_asst_idx = max(i for i, m in enumerate(msgs) if m["role"] == "assistant")
                old_text = msgs[last_asst_idx]["content"].strip()
                if old_text.upper() in {"TRUE", "FALSE"}:
                    msgs[last_asst_idx]["content"] = _make_harder_assistant_target(old_text)

            rows.append({"image_path": img_path, "messages": msgs})

    print(
        f"Loaded rows={len(rows)} | skipped(no_image)={skipped_no_image} "
        f"| skipped(no_messages)={skipped_no_msgs} | skipped(no_ground_truth)={skipped_no_gt}"
    )
    return rows


In [8]:
# -----------------------------
# Build train/eval splits
# -----------------------------
import random
random.seed(SEED)

all_rows = read_harmony_jsonl_with_images(JSONL_PATH)
print("Total rows with image_path+messages:", len(all_rows))
if len(all_rows) == 0:
    raise ValueError("No usable rows found. Verify JSONL_PATH and that records contain 'image_path' + 'messages'.")

random.shuffle(all_rows)
n_train = int(len(all_rows) * TRAIN_FRAC)
train_rows = all_rows[:n_train]
eval_rows = all_rows[n_train:]

print("Train:", len(train_rows), "Eval:", len(eval_rows))


Loaded rows=7902 | skipped(no_image)=0 | skipped(no_messages)=0 | skipped(no_ground_truth)=0
Total rows with image_path+messages: 7902
Train: 7743 Eval: 159


In [9]:
from PIL import Image
import torch
from torch.utils.data import Dataset
from typing import Any, Dict, List
from dataclasses import dataclass

# --------------------------------------------------
# Phi-4 chat rendering
# --------------------------------------------------
PHI4_SYSTEM = (
    "You are Phi, a multimodal model trained by Microsoft to help users. "
    "Provide accurate and concise responses. "
    "For this verification task, respond in the following exact format:\n"
    "Verdict: TRUE or FALSE\n"
    "Reason: a short evidence-grounded explanation."
)

def render_phi4_chat(tokenizer, messages: List[Dict[str, str]], add_generation_prompt: bool) -> str:
    msgs = messages[:]

    if len(msgs) == 0 or msgs[0].get("role") != "system":
        msgs = [{"role": "system", "content": PHI4_SYSTEM}] + msgs

    text = tokenizer.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        return_dict=False,
    )

    # Phi-4 non-thinking generation token
    if add_generation_prompt:
        text = str(text) + "<|dummy_84|>"

    return text

class HarmonyVisionTextDataset(Dataset):
    """
    Keeps ALL processor-produced tensors.
    Supervises the FULL assistant span:
      Verdict: TRUE/FALSE
      Reason: ...
    """
    def __init__(self, rows: List[Dict[str, Any]], processor, max_len: int = 1536):
        if len(rows) == 0:
            raise ValueError("HarmonyVisionTextDataset received empty rows.")
        self.rows = rows
        self.processor = processor
        self.max_len = max_len

        if not hasattr(processor, "tokenizer"):
            raise ValueError("Phi-4 processor is expected to have a tokenizer.")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.rows[idx]
        image_path = ex["image_path"]
        messages = ex["messages"]

        image = Image.open(image_path).convert("RGB")

        user_indices = [i for i, m in enumerate(messages) if m["role"] == "user"]
        if not user_indices:
            raise ValueError(f"No user turn found for row idx={idx}")
        last_user_idx = max(user_indices)

        prompt_only = messages[: last_user_idx + 1]
        full_conv = messages

        prompt_text = render_phi4_chat(
            self.processor.tokenizer,
            prompt_only,
            add_generation_prompt=True,
        )
        full_text = render_phi4_chat(
            self.processor.tokenizer,
            full_conv,
            add_generation_prompt=False,
        )

        enc_full = self.processor(
            text=full_text,
            images=[image],
            return_tensors="pt",
            truncation=True,
            max_length=self.max_len,
        )

        out: Dict[str, Any] = {}
        for k, v in enc_full.items():
            if torch.is_tensor(v):
                out[k] = v[0]
            else:
                out[k] = v

        input_ids = out["input_ids"]

        # supervise FULL assistant span
        assistant_msgs = [m for m in full_conv if m["role"] == "assistant"]
        if len(assistant_msgs) == 0:
            raise ValueError(f"No assistant message found for row idx={idx}")

        assistant_text = assistant_msgs[-1]["content"].strip()

        answer_ids = self.processor.tokenizer(
            assistant_text,
            add_special_tokens=False,
            return_tensors="pt",
        )["input_ids"][0]

        labels = torch.full_like(input_ids, -100)

        def find_subsequence(seq: torch.Tensor, subseq: torch.Tensor) -> int:
            n, m = seq.shape[0], subseq.shape[0]
            for start in range(n - m, -1, -1):  # search backward
                if torch.equal(seq[start:start + m], subseq):
                    return start
            return -1

        start = find_subsequence(input_ids, answer_ids)

        if start == -1:
            decoded_tail = self.processor.tokenizer.decode(
                input_ids[-200:].tolist(),
                skip_special_tokens=False,
            )
            raise ValueError(
                f"Could not find assistant span in input_ids for idx={idx}.\n"
                f"assistant_text={assistant_text!r}\n"
                f"decoded tail:\n{decoded_tail}"
            )

        labels[start:start + answer_ids.shape[0]] = input_ids[start:start + answer_ids.shape[0]]
        out["labels"] = labels

        return out


# -----------------------------
# Data collator
# -----------------------------
@dataclass
class VLDataCollator:
    processor: Any
    label_pad_token_id: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        tok = self.processor.tokenizer

        input_ids = [f["input_ids"] for f in features]
        attention_mask = [f.get("attention_mask", torch.ones_like(f["input_ids"])) for f in features]
        labels = [f["labels"] for f in features]

        batch = tok.pad(
            {"input_ids": input_ids, "attention_mask": attention_mask},
            padding=True,
            return_tensors="pt",
        )

        max_len = batch["input_ids"].shape[1]
        padded_labels = torch.full((len(labels), max_len), self.label_pad_token_id, dtype=torch.long)
        for i, lab in enumerate(labels):
            padded_labels[i, : lab.shape[0]] = lab
        batch["labels"] = padded_labels

        skip_keys = {"input_ids", "attention_mask", "labels"}
        extra_keys = [k for k in features[0].keys() if k not in skip_keys]

        for k in extra_keys:
            vals = [f[k] for f in features]
            if torch.is_tensor(vals[0]):
                try:
                    batch[k] = torch.stack(vals, dim=0)
                except Exception:
                    batch[k] = vals
            else:
                batch[k] = vals

        return batch

In [10]:
# -----------------------------
# Optional image token helper
#   For Phi-4, do NOT hardcode Gemma3 BOI.
#   Only insert an image placeholder if your prompt lacks one.
# -----------------------------
def ensure_image_placeholder(text: str) -> str:
    """
    Minimal safe helper.
    If your Harmony prompts already mention the image, leave them unchanged.
    Otherwise prepend a generic image marker line.
    """
    markers = [
        "<image>", "<|image_1|>", "<|image|>", "<image_1>",
        "<start_of_image>"
    ]
    if any(m in text for m in markers):
        return text
    return "<image>\n" + text

In [11]:
train_ds = HarmonyVisionTextDataset(train_rows, processor, max_len=MAX_LEN)
eval_ds = HarmonyVisionTextDataset(eval_rows, processor, max_len=MAX_LEN) if len(eval_rows) else None

data_collator = VLDataCollator(processor=processor)

sample = train_ds[0]
print({k: (v.shape, v.dtype) for k, v in sample.items() if torch.is_tensor(v)})

masked = int((sample["labels"] == -100).sum().item())
total = int(sample["labels"].numel())
print("labels masked:", masked, "/", total)
print("labels unmasked:", total - masked, "/", total)

keep_ids = sample["labels"][sample["labels"] != -100]
print("decoded supervised span:")
print(processor.tokenizer.decode(keep_ids.tolist(), skip_special_tokens=False))


{'input_ids': (torch.Size([752]), torch.int64), 'attention_mask': (torch.Size([752]), torch.int64), 'pixel_values': (torch.Size([3600, 768]), torch.float32), 'pixel_attention_mask': (torch.Size([3600]), torch.int32), 'spatial_shapes': (torch.Size([2]), torch.int64), 'labels': (torch.Size([752]), torch.int64)}
labels masked: 738 / 752
labels unmasked: 14 / 752
decoded supervised span:
Verdict: FALSE
Reason: Contradicted by the evidence.


In [ ]:
# batch = data_collator([train_ds[0], train_ds[1]])
# batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}

# model.train()
# out = model(**batch)

# print("loss:", out.loss.item())
# print("requires_grad:", out.loss.requires_grad)
# print("grad_fn:", out.loss.grad_fn)

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


loss: 3.972055196762085
requires_grad: True
grad_fn: <NllLossBackward0 object at 0x1553e74bf0d0>


In [ ]:
# sample = train_ds[0]
# input_ids = sample["input_ids"]
# labels = sample["labels"]

# label_pos = (labels != -100).nonzero(as_tuple=True)[0]
# print("label_pos:", label_pos.tolist())

# for p in label_pos.tolist():
#     print("label token:", p, processor.tokenizer.decode([labels[p].item()]))

# print("\nDecoded tail:")
# print(processor.tokenizer.decode(input_ids[-80:].tolist(), skip_special_tokens=False))

label_pos: [737]
label token: 737 FALSE

Decoded tail:
ities are present in both lower lobes Old rib fractures and pleural thickening are present on the right Heart and pulmonary  are normal
IMPRESSION: Hypoinflation with bibasilar focal atelectasis


STATEMENT:
The chest X-ray shows normal lung volumes with clear lower lobes and no atelectasis, pleural thickening, or rib fractures.<|im_end|><|im_start|>assistant<|im_sep|>FALSE<|im_end|>


In [13]:
# for row in trainer.state.log_history[:10]:
#     print(row)

In [12]:
# -----------------------------
# TrainingArguments
# -----------------------------
do_eval = eval_ds is not None and len(eval_ds) > 0

training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=10,
    save_steps=400,
    eval_steps=400 if do_eval else None,
    eval_strategy="steps" if do_eval else "no",
    save_strategy="steps",
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    fp16=False,
    gradient_checkpointing=False,   # <-- change this first
    report_to=[],
    dataloader_num_workers=2,
    optim="adamw_torch",            # <-- safer for debugging
    remove_unused_columns=False,
    seed=SEED,
)
print(training_args)


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=400,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False,
f

In [21]:
# -----------------------------
# Save LoRA adapter + processor
# -----------------------------
from peft import PeftModel
import json

import os
import json

def save_lora_adapter(model, processor, output_dir, adapter_name="lora", base_model=None):
    save_path = os.path.join(output_dir, adapter_name)
    os.makedirs(save_path, exist_ok=True)

    # 1) save LoRA adapter
    model.save_pretrained(save_path)

    # 2) save processor safely
    processor_saved = False
    processor_error = None

    try:
        # Patch missing attr for Phi4VisionRProcessor
        if not hasattr(processor, "chat_template"):
            processor.chat_template = None
        processor.save_pretrained(save_path)
        processor_saved = True
    except Exception as e:
        processor_error = str(e)

        # fallback: save tokenizer separately
        try:
            if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
                if not hasattr(processor.tokenizer, "chat_template"):
                    processor.tokenizer.chat_template = None
                processor.tokenizer.save_pretrained(save_path)
        except Exception as e_tok:
            print(f"⚠️ tokenizer save failed: {e_tok}")

        # fallback: save image processor separately if present
        for attr in ["image_processor", "feature_extractor"]:
            try:
                obj = getattr(processor, attr, None)
                if obj is not None:
                    obj.save_pretrained(save_path)
            except Exception as e_img:
                print(f"⚠️ {attr} save failed: {e_img}")

    # 3) save metadata
    meta = {
        "base_model": base_model,
        "processor_saved_directly": processor_saved,
        "processor_error": processor_error,
        "processor_class": processor.__class__.__name__,
    }

    with open(os.path.join(save_path, "adapter_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print(f"✅ LoRA adapter saved to: {save_path}")
    if processor_saved:
        print("✅ Processor saved successfully")
    else:
        print("⚠️ Processor.save_pretrained() failed; tokenizer/image processor fallback used")

In [ ]:
# test if the mask is correctly applied in the dataset

# for i in range(5):
#     sample = train_ds[i]
#     n_total = sample["labels"].numel()
#     n_masked = int((sample["labels"] == -100).sum().item())
#     n_unmasked = n_total - n_masked

#     print(f"\nSample {i}")
#     print("total:", n_total, "masked:", n_masked, "unmasked:", n_unmasked)

#     keep_ids = sample["labels"][sample["labels"] != -100]
#     print("unmasked token ids:", keep_ids.tolist())

#     if hasattr(processor, "tokenizer"):
#         print("decoded unmasked:",
#               processor.tokenizer.decode(keep_ids.tolist(), skip_special_tokens=False))


Sample 0
total: 739 masked: 738 unmasked: 1
unmasked token ids: [31451]
decoded unmasked: FALSE

Sample 1
total: 568 masked: 567 unmasked: 1
unmasked token ids: [31451]
decoded unmasked: FALSE

Sample 2
total: 797 masked: 796 unmasked: 1
unmasked token ids: [21260]
decoded unmasked: TRUE

Sample 3
total: 878 masked: 877 unmasked: 1
unmasked token ids: [21260]
decoded unmasked: TRUE

Sample 4
total: 548 masked: 547 unmasked: 1
unmasked token ids: [31451]
decoded unmasked: FALSE


In [ ]:
# -----------------------------
# Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

# Train
train_result = trainer.train()
print(train_result)

# Optional final eval
eval_metrics = None
if eval_ds is not None and len(eval_ds) > 0:
    eval_metrics = trainer.evaluate()
    print("Eval metrics:", eval_metrics)

# Save LoRA adapter + processor
save_lora_adapter(
    trainer.model,
    processor,
    OUTPUT_DIR,
    adapter_name="phi4_lora",
    base_model="microsoft/Phi-4-reasoning-vision-15B",
)

# -----------------------------
# Save summary metrics JSON
# -----------------------------
summary_json = os.path.join(OUTPUT_DIR, "train_eval_summary.json")
summary = {
    "train_metrics": train_result.metrics if hasattr(train_result, "metrics") else {},
    "eval_metrics": eval_metrics if eval_metrics is not None else {},
}
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("✅ Saved summary metrics:", summary_json)

# -----------------------------
# Save log history to CSV
# -----------------------------
import os
import csv

METRICS_CSV = os.path.join(OUTPUT_DIR, "training_metrics.csv")

log_history = trainer.state.log_history

if not log_history:
    print("⚠️ No metrics found in trainer.state.log_history")
else:

    # Sort rows by step if available
    log_history_sorted = sorted(
        log_history,
        key=lambda x: x.get("step", 0)
    )

    # Collect all keys used in metrics
    fieldnames = sorted(
        {k for row in log_history_sorted for k in row.keys()}
    )

    with open(METRICS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
            extrasaction="ignore"
        )

        writer.writeheader()

        for row in log_history_sorted:
            writer.writerow(row)

    print(f"✅ Saved metrics: {METRICS_CSV}")
    print(f"Rows saved: {len(log_history_sorted)}")

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The model is already on multiple devices. Skipping the move to device specified in `args`.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
You're using a GPT2TokenizerFast tokenizer. Please note that with a fa

Step,Training Loss,Validation Loss
400,0.000000,0.000003
800,0.000000,0.000001
1200,0.000000,0.000001
1600,0.000000,0.000000
2000,0.000000,0.000000
2400,0.000000,0.000000
2800,0.000000,0.000000
3200,0.000000,0.000000
3600,0.000000,0.000000
4000,0.000000,0.000000


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than

TrainOutput(global_step=9680, training_loss=0.01200288513740372, metrics={'train_runtime': 32187.9271, 'train_samples_per_second': 2.406, 'train_steps_per_second': 0.301, 'total_flos': 5.094817016580053e+18, 'train_loss': 0.01200288513740372, 'epoch': 10.0})


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than

Eval metrics: {'eval_loss': 3.07092768991879e-08, 'eval_runtime': 33.86, 'eval_samples_per_second': 4.696, 'eval_steps_per_second': 2.363, 'epoch': 10.0}


AttributeError: 'Phi4VisionRProcessor' object has no attribute 'chat_template'

In [ ]:
# print(type(model))
# for name, p in model.named_parameters():
#     if p.requires_grad:
#         print(name, p.shape)
#         break
    
# model.config.use_cache = False
# if hasattr(model, "enable_input_require_grads"):
#     model.enable_input_require_grads()
# batch = data_collator([train_ds[0], train_ds[1]])
# batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
# model.train()
# out = model(**batch)
# print(out.loss, out.loss.requires_grad, out.loss.grad_fn)

<class 'peft.peft_model.PeftModelForCausalLM'>
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight torch.Size([32, 5120])


You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


tensor(25.1000, device='cuda:0', grad_fn=<NllLossBackward0>) True <NllLossBackward0 object at 0x154c994014e0>


In [17]:
batch = data_collator([train_ds[0], train_ds[1]])
batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
model.train()
out = model(**batch)
print(out.loss.item(), out.loss.requires_grad)
for row in trainer.state.log_history[:10]:
    print(row)

4.257474373048353e-08 True
{'loss': 4.5167, 'grad_norm': 7.5223565101623535, 'learning_rate': 3.0927835051546395e-06, 'epoch': 0.010330578512396695, 'step': 10}
{'loss': 4.0608, 'grad_norm': 9.331011772155762, 'learning_rate': 6.529209621993128e-06, 'epoch': 0.02066115702479339, 'step': 20}
{'loss': 2.3253, 'grad_norm': 5.340196132659912, 'learning_rate': 9.965635738831616e-06, 'epoch': 0.030991735537190084, 'step': 30}
{'loss': 0.6956, 'grad_norm': 2.6275177001953125, 'learning_rate': 1.3402061855670103e-05, 'epoch': 0.04132231404958678, 'step': 40}
{'loss': 0.0187, 'grad_norm': 0.027970900759100914, 'learning_rate': 1.6838487972508592e-05, 'epoch': 0.05165289256198347, 'step': 50}
{'loss': 0.0006, 'grad_norm': 0.013897637836635113, 'learning_rate': 2.027491408934708e-05, 'epoch': 0.06198347107438017, 'step': 60}
{'loss': 0.0002, 'grad_norm': 0.0035651137586683035, 'learning_rate': 2.3711340206185567e-05, 'epoch': 0.07231404958677685, 'step': 70}
{'loss': 0.0001, 'grad_norm': 0.001932

In [18]:
import os
import json

def save_lora_adapter(model, processor, output_dir, adapter_name="lora", base_model=None):
    save_path = os.path.join(output_dir, adapter_name)
    os.makedirs(save_path, exist_ok=True)

    # 1) save LoRA adapter
    model.save_pretrained(save_path)

    # 2) save processor safely
    processor_saved = False
    processor_error = None

    try:
        # Patch missing attr for Phi4VisionRProcessor
        if not hasattr(processor, "chat_template"):
            processor.chat_template = None
        processor.save_pretrained(save_path)
        processor_saved = True
    except Exception as e:
        processor_error = str(e)

        # fallback: save tokenizer separately
        try:
            if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
                if not hasattr(processor.tokenizer, "chat_template"):
                    processor.tokenizer.chat_template = None
                processor.tokenizer.save_pretrained(save_path)
        except Exception as e_tok:
            print(f"⚠️ tokenizer save failed: {e_tok}")

        # fallback: save image processor separately if present
        for attr in ["image_processor", "feature_extractor"]:
            try:
                obj = getattr(processor, attr, None)
                if obj is not None:
                    obj.save_pretrained(save_path)
            except Exception as e_img:
                print(f"⚠️ {attr} save failed: {e_img}")

    # 3) save metadata
    meta = {
        "base_model": base_model,
        "processor_saved_directly": processor_saved,
        "processor_error": processor_error,
        "processor_class": processor.__class__.__name__,
    }

    with open(os.path.join(save_path, "adapter_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print(f"✅ LoRA adapter saved to: {save_path}")
    if processor_saved:
        print("✅ Processor saved successfully")
    else:
        print("⚠️ Processor.save_pretrained() failed; tokenizer/image processor fallback used")

In [19]:
save_lora_adapter(
    trainer.model,
    processor,
    OUTPUT_DIR,
    adapter_name="phi4_lora",
    base_model="microsoft/Phi-4-reasoning-vision-15B",
)

✅ LoRA adapter saved to: /data/liangz2/openi/Phi_4_reason/phi4_lora
⚠️ Processor.save_pretrained() failed; tokenizer/image processor fallback used


In [22]:
import os
import csv

# -----------------------------
# Save metrics to CSV
# -----------------------------
METRICS_CSV = os.path.join(OUTPUT_DIR, "training_metrics.csv")

log_history = trainer.state.log_history

if not log_history:
    print("⚠️ No metrics found in trainer.state.log_history")
else:

    # Sort rows by step if available
    log_history_sorted = sorted(
        log_history,
        key=lambda x: x.get("step", 0)
    )

    # Collect all keys used in metrics
    fieldnames = sorted(
        {k for row in log_history_sorted for k in row.keys()}
    )

    with open(METRICS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
            extrasaction="ignore"
        )

        writer.writeheader()

        for row in log_history_sorted:
            writer.writerow(row)

    print(f"✅ Saved metrics: {METRICS_CSV}")
    print(f"Rows saved: {len(log_history_sorted)}")


✅ Saved metrics: /data/liangz2/openi/Phi_4_reason/training_metrics.csv
Rows saved: 994
